# unit05 レッスン: 総合ミニプロジェクト — 汚いデータを一本のパイプラインで捌く

**このレッスンで作れるようになるもの**: 欠損だらけ・スケールがバラバラの実データを、`Pipeline`(前処理を1本に束ねる仕組み)に流し込んで **読込→前処理→分割→学習→評価** まで一気通貫で書く力。

これは unit01〜04 で個別に練習してきた道具(NumPy配列・pandas前処理・回帰・分類)を、**実務でやる形にまとめる総仕上げ**です。ここまで来れば「機械学習のミニプロジェクトを一人で回せる」状態になります。

- 所要時間: 15〜25分
- 前提: unit01〜04 を学習済み(NumPy配列・train_test_split・fit/predict・accuracy などは既知として進めます)
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
import numpy as np
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

np.set_printoptions(precision=4, suppress=True)   # 表示を見やすく(指数表記を抑制)

def check(name, actual, expected, hint=""):
    try:
        ok = actual is not None and bool(np.all(np.isclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

print("準備OK! ここからは sklearn の道具を組み合わせていきます")

---
## 概念1: 欠損補完とスケーリング — 「実データは汚い」という前提から始める

### なぜ学ぶか
unit03〜04 で使ったデータは教科書用にお膳立てされた綺麗なものでした。しかし**実務のデータはほぼ必ず汚れています**: アンケートの未回答、センサーの通信断、DBのNULL — こういう「穴(欠損)」が普通に混ざっています。さらに「年齢(0〜100)」と「年収(数百万)」のように**列ごとに数値のスケールがバラバラ**だと、多くのモデルが大きい数字の列に引きずられて正しく学習できません。だから本番のデータ分析は、いきなりモデルを作る前に**穴を埋める(欠損補完)**・**桁を揃える(スケーリング)**という下ごしらえから始めるのが定石です。

### 解説

2つの下ごしらえ道具を導入します。どちらも sklearn の **transformer(変換器)**で、`fit`(データから統計量を学ぶ)→ `transform`(その統計量で変換する)という共通の形を持ちます。

- **`SimpleImputer(strategy="mean")`**: 欠損(`np.nan`)を、その**列の平均値**で埋める変換器。`strategy` を `"median"` にすれば中央値。
- **`StandardScaler()`**: 各列を「平均0・標準偏差1」に揃える変換器(**標準化**)。これで全列が同じ土俵に乗ります。

そして最重要のルールが1つ:

> **`fit_transform` は訓練データにだけ使う。テストデータには `transform` だけ。**

`fit` は「平均や標準偏差をデータから計算する」工程です。もしテストデータまで混ぜて平均を計算してしまうと、**本番でしか見えないはずのデータの情報が前処理に漏れ込む** — これを**データリーク(データ漏洩)**と呼び、評価が不当に甘くなる典型的な事故です。「統計量は訓練データだけから学ぶ」と体に刻んでください。

まず動くコードを見ます。中間状態(欠損の数)を print で確認しながら追ってください。

In [ ]:
# GOAL: 欠損を注入 → 平均で補完 → 標準化 の3工程を、数字の変化で確認する

# STEP 1: 綺麗なデータ(load_wine)を読み、あえて欠損を注入する
#         rng.random(shape) は 0〜1 の一様乱数配列。0.1未満のセルを np.nan にすると約10%が欠損になる
wine = load_wine()
X, y = wine.data, wine.target          # X=特徴量(178行13列), y=正解ラベル(0/1/2 の3品種)
rng = np.random.default_rng(0)          # シード固定 — 何度実行しても同じ欠損パターン
X_missing = X.copy()                    # 元配列は壊さない(copyしてから書き換え)
X_missing[rng.random(X.shape) < 0.1] = np.nan
print("STEP1 欠損セルの数:", np.isnan(X_missing).sum(), "/", X.size)

# STEP 2: SimpleImputer で列ごとの平均に穴埋め(fit_transformで「平均を学ぶ+埋める」を一度に)
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_missing)
print("STEP2 補完後の欠損の数:", np.isnan(X_imputed).sum(), "(0になっていれば成功)")

# STEP 3: StandardScaler で全列を平均0・標準偏差1にそろえる
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print("STEP3 各列の平均(先頭3列):", X_scaled.mean(axis=0)[:3], "← ほぼ0")
print("STEP3 各列の標準偏差(先頭3列):", X_scaled.std(axis=0)[:3], "← ほぼ1")

### 予測してみよう

次のセルは、**訓練データとテストデータに分けてから**標準化します。ポイントは:
- 訓練データには `fit_transform`(平均・標準偏差を学びつつ変換)
- テストデータには `transform` だけ(**訓練で学んだ**平均・標準偏差を使い回す)

**実行する前に予測してください**: 訓練データの標準化後の平均はほぼ0・標準偏差はほぼ1になります。では **テストデータの標準化後の平均は、ちょうど0になるでしょうか?** ならないとしたら、それはなぜでしょう?(ヒント: テストの平均は誰の統計量で引かれている?)

In [ ]:
# 予測してから実行!
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)   # 訓練: 学ぶ+変換
X_te_s = scaler.transform(X_te)       # テスト: 訓練で学んだ統計量で変換するだけ

print("訓練データ 標準化後の平均(先頭3列):", X_tr_s.mean(axis=0)[:3], "← ほぼ0")
print("テストデータ 標準化後の平均(先頭3列):", X_te_s.mean(axis=0)[:3], "← 0ちょうどにはならない")

テストデータの平均は0ちょうどにはなりません。テストの各列は「**訓練データの**平均」で引かれているからです。これは正しい挙動です — 本番で1件データが来たとき、その1件だけで平均を計算し直すのは不可能ですよね。**前処理の統計量は訓練データに固定する**のが鉄則です。

もし逆に「テストデータにも `fit_transform` してしまったら」何がまずいか — これがまさにデータリークです。評価用のはずのテストデータの情報が前処理に混ざり、テストスコアが実力以上に良く見えてしまい、本番で通用しないモデルを「良い」と勘違いします。

### 書いてみる

**課題**: 上の `X_missing`(欠損入りのワインデータ)を、**中央値(median)**で補完してください。`SimpleImputer` の `strategy` を変えるだけです。補完後の配列を `result1` に入れてください。

チェックでは「補完後に欠損が0個になっているか(=補完後の欠損数)」を見ます。`result1` には**補完後の配列そのもの**を入れてください。期待値は「欠損数 = 0」です。

ヒント(概念レベル): `strategy="median"` の `SimpleImputer` を作り、`X_missing` を `fit_transform` する。

In [ ]:
result1 = None
# ここに書く(result1 に「中央値補完後の配列」を代入する)


# 補完後に欠損が1つも無ければ np.isnan(...).sum() は 0 になる
_missing_after = None if result1 is None else int(np.isnan(np.asarray(result1)).sum())
check("概念1: 中央値で欠損補完", _missing_after, 0,
      hint="SimpleImputer(strategy=\"median\") を作って X_missing を fit_transform する。result1 には補完後の配列を入れる")

---
## 概念2: Pipeline — 前処理とモデルを1本の管に束ねる

### なぜ学ぶか
概念1では「補完 → 標準化」を別々の変数(`X_imputed`, `X_scaled`)に手作業でつなぎました。でも工程が増えるほど、**訓練とテストで同じ順番・同じ統計量を手で揃える**のは間違いのもとです(「テストにだけ標準化を忘れた」「補完の後にスケールするはずが順番を逆にした」等)。実務では前処理の各段を **`Pipeline`(処理チェーン)**として1つのオブジェクトに合成し、`fit`/`transform` を1回呼ぶだけで全段が正しい順で流れるようにします。

### 解説

**`Pipeline`** は `[(名前, 変換器), ...]` のリストを渡して作る「処理の連結」です:

```python
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),   # 段1: 欠損補完
    ("scale",  StandardScaler()),                 # 段2: 標準化
])
```

`pipe.fit_transform(X_train)` を呼ぶと、**段1→段2 の順**でデータが流れ、各段が自動で `fit` されます。`pipe.transform(X_test)` なら、各段が**訓練時に学んだ統計量で** `transform` だけ行います。つまり Pipeline を使えば「テストには transform だけ」のルールが**自動で守られる** — データリーク防止が構造的に保証されるのが最大の利点です。

**C#アナロジー**: 複数の処理ステップを1本の**処理チェーン**に合成する発想そのものです。ASP.NET の**ミドルウェアパイプライン**(リクエストが `app.Use(...)` で登録した各段を順に通り抜ける)を思い出してください。sklearn の Pipeline も、データが各段を上から順に通り抜けます。

次のセルで、手作業(概念1)と Pipeline が**同じ結果**になることを確認します。

In [ ]:
# GOAL: 「補完→標準化」を Pipeline 1本にまとめ、手作業と同じ結果になることを確認する

# 欠損入りデータを訓練/テストに分割(ラベル y は概念1のまま)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_missing, y, test_size=0.2, random_state=0)

# STEP 1: 2段の Pipeline を組む([(名前, 変換器), ...] のリスト)
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
])

# STEP 2: 訓練データで fit_transform → 段1・段2が順に学習&変換される
Xm_tr_p = pipe.fit_transform(Xm_tr)
# テストデータは transform だけ → 訓練で学んだ統計量を使い回す(リーク防止が自動で効く)
Xm_te_p = pipe.transform(Xm_te)

print("STEP2 訓練 補完後の欠損数:", np.isnan(Xm_tr_p).sum(), " 標準偏差(先頭列):", round(float(Xm_tr_p.std(axis=0)[0]), 4))

# STEP 3: Pipeline の中身を覗く — steps に (名前, 変換器) が順に入っている
print("STEP3 ステップ数:", len(pipe.steps))
print("STEP3 ステップ名:", [name for name, _ in pipe.steps])

### 予測してみよう

次のセルでは、Pipeline に**もう1段(3段目)**を足します。`SimpleImputer` → `StandardScaler` → もう一度 `StandardScaler` という(実用上は無意味だが動作確認用の)3段構成です。

**実行する前に予測してください**: この Pipeline の `steps` の長さ(段数)はいくつになるでしょう? そして「すでに標準化された列」をもう一度 `StandardScaler` に通したら、平均と標準偏差はどう変わるでしょう?(ほぼ0・ほぼ1のものを、もう一度0・1にそろえるとどうなる?)

In [ ]:
# 予測してから実行!
pipe3 = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale1", StandardScaler()),
    ("scale2", StandardScaler()),   # 2回目の標準化
])
out = pipe3.fit_transform(Xm_tr)
print("段数(len(steps)):", len(pipe3.steps))
print("最終出力の平均(先頭3列):", out.mean(axis=0)[:3], " 標準偏差(先頭3列):", out.std(axis=0)[:3])

段数は3。すでに平均0・標準偏差1の列をもう一度標準化しても、結果はほぼ変わりません(0・1のものを0・1にそろえるだけ)。段を足せば足すほど処理は増えますが、**Pipeline なら順番と統計量の管理はすべて自動**、という感覚を掴めればOKです。

### 書いてみる

**課題**: 「欠損補完(平均)→ 標準化」の**2段**の Pipeline を自分で組み、変数 `my_pipe` に入れてください。段の名前は自由です。

チェックでは `my_pipe` の**段数(ステップ数)**を見ます。期待値は `2` です。

ヒント(概念レベル): `Pipeline([("名前1", SimpleImputer(...)), ("名前2", StandardScaler())])` の形。

In [ ]:
my_pipe = None
# ここに書く(my_pipe に 2段の Pipeline を代入する)


_n_steps = None if my_pipe is None else len(my_pipe.steps)
check("概念2: 2段のPipeline", _n_steps, 2,
      hint="Pipeline([(\"impute\", SimpleImputer(strategy=\"mean\")), (\"scale\", StandardScaler())]) の形。段は2つ")

---
## 概念3: エンドツーエンド — 1枚の地図として全体を通す

### なぜ学ぶか
ここまでの道具(補完・標準化・Pipeline)と、unit03〜04 の道具(`train_test_split`・`fit`/`predict`・`accuracy`)を**1本の流れ**につなぐと、それがそのまま**実務の最小ワークフロー**になります。新しいデータセットや案件が来ても、この地図の順番通りに手を動かせば「とりあえず動く評価済みモデル」までたどり着ける — これが本コースのゴールです。

### 解説

エンドツーエンドの流れは、いつも同じ5ステップの地図です:

```
① 読込        load_*() でデータ (X, y) を得る
② 分割        train_test_split で訓練/テストに分ける ← 分割は前処理より「先」!
③ 前処理      Pipeline を訓練で fit_transform / テストで transform
④ 学習        モデル.fit(前処理済み訓練X, 訓練y)
⑤ 評価        モデル.predict(前処理済みテストX) を正解と比べて accuracy 等を出す
```

**順番の急所は「②分割が③前処理より先」**であることです。先に分割しておかないと、前処理の統計量にテストデータが混ざり、概念1で見たデータリークが起きます。「**分けてから、下ごしらえする**」と覚えてください。

次のセルで、汚したワインデータに対してこの5ステップを最初から最後まで通します。各ステップに `# STEP` ラベルを付けたので、地図と照らし合わせて読んでください。

In [ ]:
# GOAL: 読込→分割→前処理→学習→評価 の5ステップを、汚れたワインデータで最後まで通す

# STEP 1: 読込(X_missing は概念1で作った欠損入りワイン。y はそのラベル)
#         → 分割してから前処理する順番を守る
# STEP 2: 分割(前処理より先に!)
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(X_missing, y, test_size=0.2, random_state=0)

# STEP 3: 前処理(訓練で fit_transform / テストで transform)
prep = Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())])
Xe_tr_p = prep.fit_transform(Xe_tr)
Xe_te_p = prep.transform(Xe_te)

# STEP 4: 学習(LogisticRegression。max_iter は収束のため大きめに)
model = LogisticRegression(max_iter=5000, random_state=0)
model.fit(Xe_tr_p, ye_tr)

# STEP 5: 評価(テストを予測 → 正解率と混同行列)
pred = model.predict(Xe_te_p)
acc = accuracy_score(ye_te, pred)
print("STEP5 テスト正解率(accuracy):", round(float(acc), 4))
print("STEP5 混同行列(行=正解, 列=予測):\n", confusion_matrix(ye_te, pred))

### 予測してみよう

次のセルは、まったく同じ5ステップの地図を **別のデータセット(`load_breast_cancer`、乳がんの良性/悪性の2クラス分類)** に適用します。欠損率は5%で注入します。

**実行する前に予測してください**: データセットが変わっても、コードの**構造(5ステップの順番)**は1文字も変える必要がありません。正解率(accuracy)は 0.5(でたらめ)より明確に高く、0.9前後の高い値になるはずです。なぜ「構造を変えずに」新しいデータに適用できるのでしょう?(ヒント: 各ステップは X, y の中身に依存しない汎用の道具でしたね)

In [ ]:
# 予測してから実行!
# ① 読込
bc = load_breast_cancer()
Xb, yb = bc.data, bc.target
# 欠損5%を注入(シード固定)
Xb_missing = Xb.copy()
Xb_missing[np.random.default_rng(0).random(Xb.shape) < 0.05] = np.nan

# ② 分割 → ③ 前処理 → ④ 学習 → ⑤ 評価(構造は前セルと同一)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb_missing, yb, test_size=0.2, random_state=0)
prep_b = Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())])
Xb_tr_p = prep_b.fit_transform(Xb_tr)
Xb_te_p = prep_b.transform(Xb_te)
model_b = LogisticRegression(max_iter=5000, random_state=0)
model_b.fit(Xb_tr_p, yb_tr)
acc_b = accuracy_score(yb_te, model_b.predict(Xb_te_p))
print("breast_cancer のテスト正解率:", round(float(acc_b), 4))

同じ地図が、中身の違うデータにそのまま通用しました。各ステップ(分割・補完・標準化・学習・評価)は X, y の具体的な中身に依存しない**汎用の道具**なので、データが変わっても順番通りに並べるだけで動きます。これが「エンドツーエンドの流れを一度身につければ、案件が変わっても戦える」という意味です。

### 書いてみる

**課題**: 上の `breast_cancer` の5ステップを**あなたの手でもう一度**書き、**テストの正解率**を `result3` に入れてください。データ・分割条件・Pipeline構成・モデルは上のセルと**まったく同じ**(欠損率0.05・`random_state=0`・`test_size=0.2`・`max_iter=5000`)にしてください。そうすれば結果は再現します。

チェックでは `result3`(正解率)が期待値と一致するかを見ます。期待値は `0.9737`(小数第4位まで)です。

ヒント(概念レベル): 上の「予測してみよう」のセルとまったく同じ手順。`accuracy_score(...)` の結果を `result3` に入れるだけ。

In [ ]:
result3 = None
# ここに書く(result3 に breast_cancer のテスト正解率を代入する)
# 手順: 読込→欠損注入(0.05, seed0)→分割(test_size=0.2, random_state=0)
#       →Pipeline(impute mean + scale)で前処理→LogisticRegression(max_iter=5000, random_state=0)で学習→accuracy


check("概念3: エンドツーエンド正解率", None if result3 is None else round(float(result3), 4), 0.9737,
      hint="前セルと完全に同じ手順。np.random.default_rng(0)・test_size=0.2・random_state=0・max_iter=5000 をそろえれば再現する")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:
- **`fit_transform` を訓練データだけに使う理由を、一言で**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | 急所 |
|------|--------|------|
| 欠損補完 & スケーリング | `SimpleImputer` で穴埋め、`StandardScaler` で桁揃え | **fit_transform は訓練だけ**・テストは transform(データリーク防止) |
| Pipeline | 前処理を `[(名前, 変換器), ...]` で1本に合成 | ミドルウェアパイプライン同様、データが各段を順に通る。リーク防止が自動 |
| エンドツーエンド | 読込→**分割**→前処理→学習→評価 の5ステップの地図 | **分けてから下ごしらえ**(分割が前処理より先) |

**この流れがそのまま実務の最小ワークフローです**。案件やデータセットが変わっても、この5ステップの地図を順に辿れば「評価済みの動くモデル」までたどり着けます。unit01(NumPy)→ unit02(pandas前処理)→ unit03(回帰)→ unit04(分類・評価)で仕込んだ道具が、ここで1本につながりました。

**この先どこで使うか**: 次のステップは「自分でデータを集める」段階です。Webスクレイピングや API でデータを取得し(次コース候補)、そのままこの5ステップに流し込めば、あなただけのデータでモデルが作れます。今日の地図はどんなデータにも効く共通の骨格です。

**次**: 演習 `ex01_preprocess.py` から順に進もう。lesson を見ながらで OK。テストは
`python -m pytest courses/ml-intro/unit05-capstone-project/tests/test_ex01.py -q`